# 実TEM画像のノイズ解析 — 視覚的デモ

このノートブックは、**実際のTEM（透過型電子顕微鏡）画像**からノイズの統計的特性を抽出し、
その特性を再現する合成ノイズを生成する **5ステップのパイプライン** を視覚的に説明します。

## 5ステップパイプライン

| ステップ | 内容 | 目的 |
|---------|------|------|
| **Step 1** | 画像・マスク読み込み | ナノシートと背景領域を分離 |
| **Step 2** | 低周波・高周波分解 | ガウスフィルタで構造とノイズを分離 |
| **Step 3** | ノイズ統計量の計算 | 平均・標準偏差・パワースペクトル密度(PSD) |
| **Step 4** | 複数画像の集約 | ロバストなノイズモデルを構築 |
| **Step 5** | ノイズの合成・再現 | 統計的に等価な合成ノイズを生成 |

> **前提知識**: SNR の概念は `01_SNR.ipynb` および `02_SNR_image.ipynb` を参照してください。

---
## 1 — セットアップ

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
%matplotlib inline

# scipy は標準的な科学計算環境に含まれる
from scipy.ndimage import gaussian_filter

# パス設定 / Path configuration
DATA_DIR = Path("../data/C14_Clay_BA_cropped")
GT_DIR   = DATA_DIR / "ground_truth"

# ハイパーパラメータ / Hyperparameters
SIGMA      = 50    # Gaussian blur sigma for low-freq separation (pixels)
PATCH_SIZE = 256   # patch size for FFT analysis

# 再現性のための乱数シード / Seeded RNG for reproducibility
RNG = np.random.default_rng(42)

print("Setup complete.")
print(f"DATA_DIR : {DATA_DIR.resolve()}")
print(f"GT_DIR   : {GT_DIR.resolve()}")

---
## 2 — Step 1: 実TEM画像とマスクの読み込み

### TEM画像の読み込み

TEM画像は **16ビットTIFFファイル** として保存されています。  
`tifffile` ライブラリが利用可能な場合はそちらを優先し、  
なければ `PIL.Image` にフォールバックします。  
読み込み後は **[0, 1] に正規化** します。

### マスクの読み込み

グランドトゥルースマスクは **NPZファイル**（NumPy圧縮形式）に保存されています。  
各ファイルには**インスタンスマスク**（個々のナノシートの領域）が格納されており、  
これらを OR 結合して「ナノシート全体のバイナリマスク」を作成します。

```
mask_nanosheet = 全ナノシートのOR結合 → True = ナノシート部分
mask_bg        = ~mask_nanosheet       → True = 背景部分（ノイズ解析に使用）
```

In [ ]:
# --- 画像・マスク読み込みヘルパー関数 / Image & mask loading helpers ---

def load_normalize(path):
    """Load a TIF image and normalize to [0, 1] float64."""
    try:
        import tifffile
        arr = tifffile.imread(str(path))
    except ImportError:
        from PIL import Image
        arr = np.array(Image.open(str(path)))
    arr = arr.astype(np.float64)
    if arr.ndim == 3:          # handle RGB or multi-page
        arr = arr[..., 0] if arr.shape[-1] <= 4 else arr[0]
    arr = (arr - arr.min()) / (arr.max() - arr.min() + 1e-12)
    return arr


def load_mask(npz_path):
    """Load NPZ ground-truth masks and return combined boolean mask.
    
    Tries common key names; falls back to inspecting all arrays.
    Returns True where nanosheet pixels are present.
    """
    data = np.load(str(npz_path), allow_pickle=True)
    keys = list(data.files)

    # Try well-known key names first
    candidate_keys = ['masks', 'mask', 'segmentation', 'arr_0']
    found_key = None
    for k in candidate_keys:
        if k in keys:
            found_key = k
            break
    if found_key is None:
        found_key = keys[0]  # fall back to first available key

    arr = data[found_key]
    # arr may be: (N, H, W) instance masks, (H, W) label map, or (H, W) bool
    if arr.ndim == 3:
        combined = np.any(arr.astype(bool), axis=0)
    elif arr.ndim == 2:
        combined = arr.astype(bool)
    else:
        raise ValueError(f"Unexpected mask shape {arr.shape} in {npz_path}")
    return combined


# --- 画像 0010 を読み込んで確認 / Load image 0010 for inspection ---
img_path_0010  = DATA_DIR / "C14-Clay_BA_x3-0010_cropped.tif"
mask_path_0010 = GT_DIR   / "C14-Clay_BA_x3-0010.npz"

img          = load_normalize(img_path_0010)
mask_nano    = load_mask(mask_path_0010)
mask_bg      = ~mask_nano

print(f"Image shape : {img.shape}  dtype: {img.dtype}")
print(f"Value range : [{img.min():.4f}, {img.max():.4f}]")
print(f"Nanosheet pixels : {mask_nano.sum():,} ({100*mask_nano.mean():.1f}%)")
print(f"Background pixels: {mask_bg.sum():,} ({100*mask_bg.mean():.1f}%)")

# --- 3パネル可視化 / 3-panel visualization ---
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(img, cmap='gray', vmin=0, vmax=1)
axes[0].set_title('TEM画像 (0010)\nOriginal TEM Image', fontsize=11)

axes[1].imshow(mask_nano, cmap='Reds', vmin=0, vmax=1)
axes[1].set_title('ナノシートマスク\nNanosheet Mask', fontsize=11)

bg_vis = img.copy()
bg_vis[mask_nano] = np.nan
axes[2].imshow(bg_vis, cmap='gray', vmin=0, vmax=1)
axes[2].set_title('背景領域のみ\nBackground Region Only', fontsize=11)

for ax in axes:
    ax.axis('off')

fig.suptitle('Step 1 — 画像とマスクの読み込み / Image & Mask Loading', fontsize=13)
plt.tight_layout()
plt.show()

### Step 1 まとめ

- **TEM画像**: 16ビットTIFFを読み込み、[0,1]に正規化
- **NPZマスク**: インスタンスマスクをOR結合して「ナノシート / 背景」の2値マスクを生成
- **背景領域** (mask_bg) のみを使ってノイズ統計を推定する → ナノシートの構造に汚染されない純粋なノイズを観測できる

> **Key insight**: ナノシート部分はノイズ以外の構造信号を含むため、ノイズ推定から除外する必要があります。

---
## 3 — Step 2: 低周波・高周波分解

TEM画像 $I(x,y)$ は大きく2つの成分に分解できます：

$$I(x,y) = L(x,y) + \eta(x,y)$$

- $L(x,y)$: **低周波成分**（ナノシートの輝度プロファイル、電子ビームの不均一性など）  
  → ガウスフィルタで近似: $L = G_\sigma * I$
- $\eta(x,y)$: **高周波成分**（ノイズ）  
  → 差し引きで取得: $\eta = I - L$

### σ の選び方

| σ が小さすぎる | σ が大きすぎる |
|--------------|---------------|
| L にノイズが残る | L にナノシートの細部が消える |
| η が小さく推定される | η に構造信号が混入する |

ここでは `SIGMA = 50` ピクセルを使用します（ナノシートサイズより大きく、画像全体サイズより小さい値）。

In [ ]:
# --- 低周波・高周波分解 / Low-freq / high-freq decomposition ---
low_freq  = gaussian_filter(img, sigma=SIGMA)
high_freq = img - low_freq

print(f"low_freq  : mean={low_freq.mean():.4f}  std={low_freq.std():.4f}  "
      f"range=[{low_freq.min():.4f}, {low_freq.max():.4f}]")
print(f"high_freq : mean={high_freq.mean():.6f}  std={high_freq.std():.4f}  "
      f"range=[{high_freq.min():.4f}, {high_freq.max():.4f}]")

# η のヒストグラム / Histogram of high-freq component
eta_vals = high_freq.ravel()
eta_lim  = np.percentile(np.abs(eta_vals), 99.5)

# --- 4パネル可視化 / 4-panel visualization ---
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

im0 = axes[0].imshow(img, cmap='gray', vmin=0, vmax=1)
axes[0].set_title('原画像 $I$\nOriginal', fontsize=11)
plt.colorbar(im0, ax=axes[0], fraction=0.046)

im1 = axes[1].imshow(low_freq, cmap='gray', vmin=low_freq.min(), vmax=low_freq.max())
axes[1].set_title(f'低周波成分 $L$ (σ={SIGMA})\nLow-freq (Gaussian blur)', fontsize=11)
plt.colorbar(im1, ax=axes[1], fraction=0.046)

# Diverging colormap for residual (centered at 0)
im2 = axes[2].imshow(high_freq, cmap='RdBu_r', vmin=-eta_lim, vmax=eta_lim)
axes[2].set_title('高周波成分 $η = I - L$\nHigh-freq (noise residual)', fontsize=11)
plt.colorbar(im2, ax=axes[2], fraction=0.046)

axes[3].hist(eta_vals, bins=200, range=(-eta_lim, eta_lim),
             color='steelblue', edgecolor='none', density=True)
# ガウス分布フィット / Gaussian fit overlay
mu, sigma_est = eta_vals.mean(), eta_vals.std()
x_fit = np.linspace(-eta_lim, eta_lim, 300)
axes[3].plot(x_fit, np.exp(-0.5*((x_fit-mu)/sigma_est)**2) / (sigma_est*np.sqrt(2*np.pi)),
             color='tomato', lw=2, label=f'Gaussian fit\nμ={mu:.4f}, σ={sigma_est:.4f}')
axes[3].set_title('η のヒストグラム\nHistogram of η', fontsize=11)
axes[3].set_xlabel('Pixel value')
axes[3].set_ylabel('Density')
axes[3].legend(fontsize=9)
axes[3].grid(True, alpha=0.3)

for ax in axes[:3]:
    ax.axis('off')

fig.suptitle('Step 2 — 低周波・高周波分解 / Low-freq & High-freq Decomposition', fontsize=13)
plt.tight_layout()
plt.show()

### Step 2 まとめ

- **低周波成分** $L$: ガウスブラー（σ=50）でゆっくりとした輝度変化を捉える
- **高周波成分** $\eta = I - L$: ほぼゼロ平均のランダム成分 → **ノイズ**
- ヒストグラムがガウス分布に近い → 白色ガウスノイズの仮定が成り立つ

> **Key insight**: σ=50 は「ナノシートよりは大きいが電子ビーム不均一性よりは小さい」スケールを選んでいます。

---
## 4 — Step 3: 高周波ノイズの統計量

高周波成分 $\eta$ の統計量を **背景領域のみ** で計算します：

| 統計量 | 意味 |
|--------|------|
| `hf_mean ≈ 0` | ノイズはゼロ平均（系統的バイアスなし） |
| `hf_std` | ノイズの強度（大きいほどノイジー） |

### パワースペクトル密度 (PSD)

ノイズが**白色** (white noise) か **有色** (colored noise) かを調べるには
パワースペクトル密度を計算します：

$$\text{PSD}(k) = \frac{1}{N} |\mathcal{F}\{\eta\}(k)|^2$$

2D FFT → 空間周波数の動径方向に平均 → **動径 PSD 曲線** を得ます。  
白色ノイズなら PSD は平坦、有色ノイズなら $1/f^\alpha$ 的な形状になります。

In [ ]:
# --- 背景領域のノイズ統計 / Noise statistics from background region ---
eta_bg   = high_freq[mask_bg]
hf_mean  = eta_bg.mean()
hf_std   = eta_bg.std()
print(f"hf_mean = {hf_mean:.6f}  (≈0 expected)")
print(f"hf_std  = {hf_std:.6f}")


# --- パッチ抽出 / Extract background patches ---
def extract_background_patches(hf_img, bg_mask, patch_size, bg_thresh=0.8):
    """Slide a patch_size×patch_size window; keep patches with >bg_thresh background."""
    H, W = hf_img.shape
    patches = []
    stride  = patch_size // 2
    for y in range(0, H - patch_size + 1, stride):
        for x in range(0, W - patch_size + 1, stride):
            m = bg_mask[y:y+patch_size, x:x+patch_size]
            if m.mean() > bg_thresh:
                patches.append(hf_img[y:y+patch_size, x:x+patch_size])
    return patches


# --- 動径 PSD 計算 / Radial PSD computation ---
def compute_radial_psd(patch):
    """Return (freqs_normalized, radial_psd) for a square patch."""
    N     = patch.shape[0]
    ft    = np.fft.fftshift(np.fft.fft2(patch))
    power = np.abs(ft) ** 2
    cx, cy = N // 2, N // 2
    ys, xs = np.ogrid[:N, :N]
    r_map  = np.sqrt((ys - cy)**2 + (xs - cx)**2).astype(int)
    r_flat = r_map.ravel()
    p_flat = power.ravel()
    max_r  = r_flat.max() + 1
    psd    = np.bincount(r_flat, weights=p_flat) / np.bincount(r_flat)
    freqs  = np.arange(len(psd)) / N
    return freqs, psd


patches = extract_background_patches(high_freq, mask_bg, PATCH_SIZE)
print(f"\nExtracted {len(patches)} background patches ({PATCH_SIZE}×{PATCH_SIZE})")

if patches:
    freqs, psd_example = compute_radial_psd(patches[0])
    # 2D パワースペクトル (1枚目のパッチ) / 2D power spectrum of first patch
    ft_2d    = np.fft.fftshift(np.fft.fft2(patches[0]))
    psd_2d   = np.log1p(np.abs(ft_2d)**2)

    # --- 3パネル可視化 / 3-panel visualization ---
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    # 背景のみのη / η for background only
    eta_bg_img = np.where(mask_bg, high_freq, np.nan)
    eta_lim_bg = np.nanpercentile(np.abs(high_freq[mask_bg]), 99)
    im0 = axes[0].imshow(eta_bg_img, cmap='RdBu_r', vmin=-eta_lim_bg, vmax=eta_lim_bg)
    axes[0].set_title('背景領域の η\nη (background only)', fontsize=11)
    plt.colorbar(im0, ax=axes[0], fraction=0.046)
    axes[0].axis('off')

    # 2D PSD of one patch
    im1 = axes[1].imshow(psd_2d, cmap='viridis')
    axes[1].set_title(f'パッチのパワースペクトル\n2D PSD (patch #{0})', fontsize=11)
    plt.colorbar(im1, ax=axes[1], fraction=0.046)
    axes[1].axis('off')

    # 動径 PSD 曲線 / Radial PSD curve
    axes[2].semilogy(freqs[1:], psd_example[1:], color='steelblue', lw=1.5)
    axes[2].set_title('動径PSD曲線\nRadial PSD (single patch)', fontsize=11)
    axes[2].set_xlabel('Normalized spatial frequency')
    axes[2].set_ylabel('Power (log scale)')
    axes[2].grid(True, which='both', alpha=0.3)

    fig.suptitle('Step 3 — ノイズ統計量と PSD / Noise Statistics & PSD', fontsize=13)
    plt.tight_layout()
    plt.show()
else:
    print("Warning: no background patches found — try reducing bg_thresh or PATCH_SIZE.")

---
## 5 — Step 4: 複数画像の集約

1枚の画像からの推定は統計的にばらつきます。  
グランドトゥルースマスクが利用可能な **5枚の画像 (0010, 0012, 0014, 0015, 0019)** を使って
ノイズモデルを集約します：

- 各画像の `hf_mean`, `hf_std`, `bg_mean`, `bg_std` を記録  
- 各画像の **動径PSD** を計算し平均化 → **平均PSD曲線**  
- これらをまとめた `noise_model` 辞書を構築

In [ ]:
# --- マスクありの画像リスト / Image files with available GT masks ---
GT_IDS = ['0010', '0012', '0014', '0015', '0019']

image_files = [DATA_DIR / f"C14-Clay_BA_x3-{gid}_cropped.tif" for gid in GT_IDS]
mask_files  = [GT_DIR   / f"C14-Clay_BA_x3-{gid}.npz"         for gid in GT_IDS]

# 確認 / Verify files exist
for f in image_files + mask_files:
    if not f.exists():
        print(f"WARNING: missing {f}")

# --- 集約ループ / Aggregation loop ---
all_psds   = []   # list of (freqs, psd) per image
stats_list = []   # list of dicts
profiles   = []   # low_freq images

for gid, img_path, msk_path in zip(GT_IDS, image_files, mask_files):
    _img       = load_normalize(img_path)
    _mask_nano = load_mask(msk_path)
    _mask_bg   = ~_mask_nano
    _low_freq  = gaussian_filter(_img, sigma=SIGMA)
    _high_freq = _img - _low_freq
    _eta_bg    = _high_freq[_mask_bg]

    stats_list.append({
        'id'      : gid,
        'hf_mean' : _eta_bg.mean(),
        'hf_std'  : _eta_bg.std(),
        'bg_mean' : _img[_mask_bg].mean(),
        'bg_std'  : _img[_mask_bg].std(),
        'bg_frac' : _mask_bg.mean(),
    })

    _patches = extract_background_patches(_high_freq, _mask_bg, PATCH_SIZE)
    if _patches:
        # Average PSD over all patches from this image
        _psds = [compute_radial_psd(p) for p in _patches]
        _avg_psd = np.mean([p for _, p in _psds], axis=0)
        _freqs   = _psds[0][0]
        all_psds.append((_freqs, _avg_psd))
    else:
        all_psds.append((None, None))

    profiles.append(_low_freq)
    print(f"  [{gid}]  hf_mean={stats_list[-1]['hf_mean']:+.6f}  "
          f"hf_std={stats_list[-1]['hf_std']:.6f}  "
          f"patches={len(_patches)}")

# --- ノイズモデルの構築 / Build noise model ---
valid_psds = [(f, p) for f, p in all_psds if p is not None]
mean_psd   = np.mean([p for _, p in valid_psds], axis=0)
ref_freqs  = valid_psds[0][0]

noise_model = {
    'bg_mean'        : np.mean([s['bg_mean']  for s in stats_list]),
    'bg_std'         : np.mean([s['bg_std']   for s in stats_list]),
    'hf_mean'        : np.mean([s['hf_mean']  for s in stats_list]),
    'hf_std'         : np.mean([s['hf_std']   for s in stats_list]),
    'mean_radial_psd': mean_psd,
    'ref_freqs'      : ref_freqs,
    'profiles'       : profiles,
}

# --- モデルサマリー表示 / Print model summary ---
print("\n=== Noise Model Summary ===")
print(f"  bg_mean  = {noise_model['bg_mean']:.4f}  (background mean intensity)")
print(f"  bg_std   = {noise_model['bg_std']:.4f}  (background intensity std)")
print(f"  hf_mean  = {noise_model['hf_mean']:+.6f}  (high-freq mean, expect ≈0)")
print(f"  hf_std   = {noise_model['hf_std']:.6f}  (noise strength)")
print()
print(f"{'ID':>6}  {'bg_mean':>8}  {'bg_std':>8}  {'hf_mean':>10}  {'hf_std':>8}  {'bg_frac':>8}")
print('-' * 58)
for s in stats_list:
    print(f"  {s['id']}  {s['bg_mean']:8.4f}  {s['bg_std']:8.4f}  "
          f"{s['hf_mean']:+10.6f}  {s['hf_std']:8.6f}  {s['bg_frac']:8.3f}")

# --- PSD プロット / PSD plot: individual (thin) + mean (bold) ---
fig, ax = plt.subplots(figsize=(10, 5))
colors_per_img = plt.cm.tab10(np.linspace(0, 0.8, len(GT_IDS)))

for (f, p), gid, c in zip(all_psds, GT_IDS, colors_per_img):
    if p is not None:
        ax.semilogy(f[1:], p[1:], lw=0.8, alpha=0.5, color=c, label=f'img {gid}')

ax.semilogy(ref_freqs[1:], mean_psd[1:],
            lw=2.5, color='black', label='Mean PSD')

ax.set_title('Step 4 — 動径 PSD (各画像 + 平均)\nRadial PSD per image + mean', fontsize=12)
ax.set_xlabel('Normalized spatial frequency')
ax.set_ylabel('Power (log scale)')
ax.legend(fontsize=9, loc='upper right')
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()

---
## 6 — Step 5: ノイズの合成・再現

構築したノイズモデルから**統計的に等価な合成ノイズ**を生成します：

$$\text{合成ノイズ} = \mathcal{F}^{-1}\left[ \mathcal{F}\{w\} \cdot \sqrt{\text{PSD}(r)} \right]$$

ここで $w \sim \mathcal{N}(0,1)$ は白色ガウスノイズです。  

### アルゴリズム

1. 白色ノイズ $w$ を生成
2. FFT で周波数領域に変換
3. 平均 PSD の平方根でフィルタリング（周波数ごとに重みを調整）
4. 逆 FFT で空間領域に戻す
5. `hf_std` でスケールを正規化

In [ ]:
# --- ノイズ合成関数 / Noise synthesis function ---
def synthesize_noise(shape, noise_model, rng):
    """Generate colored noise matching the noise_model PSD and hf_std."""
    H, W      = shape
    white     = rng.normal(0, 1, shape)
    ft        = np.fft.fftshift(np.fft.fft2(white))

    # Build 2D radial index map
    cx, cy    = H // 2, W // 2
    ys, xs    = np.ogrid[:H, :W]
    r_map     = np.sqrt((ys - cy)**2 + (xs - cx)**2).astype(int)
    mean_psd  = noise_model['mean_radial_psd']
    r_map     = np.clip(r_map, 0, len(mean_psd) - 1)

    # Apply PSD filter in frequency domain
    filt      = np.sqrt(mean_psd[r_map])
    colored_ft = ft * filt

    noise = np.real(np.fft.ifft2(np.fft.ifftshift(colored_ft)))
    # Normalize to match observed hf_std
    noise = noise / (noise.std() + 1e-12) * noise_model['hf_std']
    return noise


# 最後に読み込んだ画像を使って比較 / Use image 0010 for comparison
img_ref    = load_normalize(image_files[0])
mask_nano_ref = load_mask(mask_files[0])
mask_bg_ref   = ~mask_nano_ref
lf_ref     = gaussian_filter(img_ref, sigma=SIGMA)
hf_ref     = img_ref - lf_ref

synth = synthesize_noise(img_ref.shape, noise_model, RNG)

eta_lim_ref = np.percentile(np.abs(hf_ref[mask_bg_ref]), 99)

# --- 4パネル + PSD比較 / 4-panel + PSD overlay ---
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Row 1: spatial domain
im0 = axes[0,0].imshow(hf_ref, cmap='RdBu_r', vmin=-eta_lim_ref, vmax=eta_lim_ref)
axes[0,0].set_title('実η (高周波ノイズ)\nReal η', fontsize=11)
plt.colorbar(im0, ax=axes[0,0], fraction=0.046)
axes[0,0].axis('off')

im1 = axes[0,1].imshow(synth, cmap='RdBu_r', vmin=-eta_lim_ref, vmax=eta_lim_ref)
axes[0,1].set_title('合成ノイズ\nSynthetic noise', fontsize=11)
plt.colorbar(im1, ax=axes[0,1], fraction=0.046)
axes[0,1].axis('off')

# Histogram comparison
axes[0,2].hist(hf_ref[mask_bg_ref], bins=150, range=(-eta_lim_ref, eta_lim_ref),
               density=True, alpha=0.6, color='steelblue', label='Real η (bg)')
axes[0,2].hist(synth.ravel(), bins=150, range=(-eta_lim_ref, eta_lim_ref),
               density=True, alpha=0.6, color='tomato', label='Synthetic')
axes[0,2].set_title('ヒストグラム比較\nHistogram comparison', fontsize=11)
axes[0,2].set_xlabel('Pixel value')
axes[0,2].set_ylabel('Density')
axes[0,2].legend(fontsize=9)
axes[0,2].grid(True, alpha=0.3)

# Row 2: 2D PSD comparison
psd_real_2d  = np.log1p(np.abs(np.fft.fftshift(np.fft.fft2(hf_ref)))**2)
psd_synth_2d = np.log1p(np.abs(np.fft.fftshift(np.fft.fft2(synth)))**2)

im2 = axes[1,0].imshow(psd_real_2d, cmap='viridis')
axes[1,0].set_title('実η の2D PSD\n2D PSD of real η', fontsize=11)
plt.colorbar(im2, ax=axes[1,0], fraction=0.046)
axes[1,0].axis('off')

im3 = axes[1,1].imshow(psd_synth_2d, cmap='viridis')
axes[1,1].set_title('合成ノイズの2D PSD\n2D PSD of synthetic', fontsize=11)
plt.colorbar(im3, ax=axes[1,1], fraction=0.046)
axes[1,1].axis('off')

# Radial PSD overlay
if patches:  # patches from image 0010
    patches_ref = extract_background_patches(hf_ref, mask_bg_ref, PATCH_SIZE)
    patches_syn = extract_background_patches(synth, np.ones_like(mask_bg_ref), PATCH_SIZE)

    if patches_ref:
        f_r, p_r = np.mean([compute_radial_psd(p) for p in patches_ref], axis=0)
        axes[1,2].semilogy(f_r[1:], p_r[1:], color='steelblue', lw=1.8, label='Real η')
    if patches_syn:
        f_s, p_s = np.mean([compute_radial_psd(p) for p in patches_syn], axis=0)
        axes[1,2].semilogy(f_s[1:], p_s[1:], color='tomato', lw=1.8, ls='--', label='Synthetic')
    axes[1,2].semilogy(ref_freqs[1:], mean_psd[1:], color='black',
                       lw=1.2, ls=':', label='Model mean PSD')

axes[1,2].set_title('動径 PSD オーバーレイ\nRadial PSD overlay', fontsize=11)
axes[1,2].set_xlabel('Normalized spatial frequency')
axes[1,2].set_ylabel('Power (log scale)')
axes[1,2].legend(fontsize=9)
axes[1,2].grid(True, which='both', alpha=0.3)

fig.suptitle('Step 5 — ノイズの合成・再現 / Noise Synthesis & Validation', fontsize=13)
plt.tight_layout()
plt.show()

# 統計比較表 / Statistics comparison table
print("\n=== Statistics Comparison ===")
print(f"{'Metric':>15}  {'Real η (bg)':>12}  {'Synthetic':>12}")
print('-' * 44)
real_vals  = hf_ref[mask_bg_ref]
synth_vals = synth.ravel()
for label, r, s in [
    ('mean',   real_vals.mean(),   synth_vals.mean()),
    ('std',    real_vals.std(),    synth_vals.std()),
    ('min',    real_vals.min(),    synth_vals.min()),
    ('max',    real_vals.max(),    synth_vals.max()),
]:
    print(f"  {label:>13}  {r:>12.6f}  {s:>12.6f}")

---
## 7 — インタラクティブ比較 (Optional)

`ipywidgets` がインストールされている場合、σ スライダーでリアルタイムに分解結果を確認できます。  
σ を変えると低周波成分と高周波成分がどのように変化するかを観察してください。

In [ ]:
try:
    from ipywidgets import interact, IntSlider
    _rng_widget = np.random.default_rng(0)

    def compare(sigma=50):
        low  = gaussian_filter(img_ref, sigma=sigma)
        eta  = img_ref - low
        syn  = synthesize_noise(img_ref.shape, noise_model, np.random.default_rng(sigma))

        eta_lim = np.percentile(np.abs(eta), 99)
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))

        axes[0].imshow(img_ref, cmap='gray', vmin=0, vmax=1)
        axes[0].set_title('Original TEM image', fontsize=10)
        axes[0].axis('off')

        axes[1].imshow(low, cmap='gray')
        axes[1].set_title(f'Low-freq L (σ={sigma})', fontsize=10)
        axes[1].axis('off')

        axes[2].hist(eta[mask_bg_ref], bins=100, range=(-eta_lim, eta_lim),
                     density=True, alpha=0.6, color='steelblue', label=f'Real η (σ={sigma})')
        axes[2].hist(syn.ravel(), bins=100, range=(-eta_lim, eta_lim),
                     density=True, alpha=0.6, color='tomato', label='Synthetic')
        axes[2].set_title('η vs synthetic histogram', fontsize=10)
        axes[2].legend(fontsize=9)
        axes[2].grid(True, alpha=0.3)

        print(f"σ={sigma}: hf_std(real)={eta[mask_bg_ref].std():.5f}  "
              f"hf_std(synth)={syn.std():.5f}")
        plt.tight_layout()
        plt.show()

    interact(compare, sigma=IntSlider(min=5, max=100, step=5, value=50,
                                      description='σ (blur)',
                                      continuous_update=False))
except ImportError:
    print("ipywidgets not available — skipping interactive demo.")
    print("Install with:  pip install ipywidgets")

---
## 8 — まとめ / Summary

### 5ステップ パイプライン まとめ

| ステップ | 操作 | 出力 | 検証 |
|---------|------|------|------|
| **Step 1** | TIF読み込み + NPZマスク | `img`, `mask_bg` | 背景領域がナノシートを除外していること |
| **Step 2** | ガウスフィルタ分解 | `low_freq`, `high_freq` | η ヒストグラムがガウス分布に近い |
| **Step 3** | 背景領域の統計 + PSD | `hf_mean≈0`, `hf_std`, 動径PSD | `hf_mean < 0.001` |
| **Step 4** | 5画像の集約 | `noise_model` dict | PSD曲線が滑らかで単調減少 |
| **Step 5** | PSDフィルタリングで合成 | `synth` | ヒストグラム・PSD曲線が一致 |

### 重要な知見 / Key insights

1. **TEM ノイズはほぼガウス分布** — ショットノイズ + 読み出しノイズの合成  
2. **有色ノイズ** — PSD が $1/f^\alpha$ 的な形状（完全な白色ではない）  
3. **背景マスクの重要性** — ナノシート除外なしでは構造信号がノイズ推定を汚染する  
4. **σ の感度** — σ が小さすぎるとノイズが過小評価、大きすぎると構造がηに混入する  
5. **合成ノイズの検証** — ヒストグラムと動径 PSD の両方が一致することを確認する